In [1]:
# =========================
# Core Python
# =========================
import os
import random
import numpy as np
import pandas as pd
from pathlib import Path

# =========================
# PyTorch
# =========================
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torch.optim as optim
from tqdm import tqdm
from transformers import CLIPModel
from torch.cuda.amp import autocast
from torch.amp import GradScaler   # new recommended API
# =========================
# Vision
# =========================
from PIL import Image
import torchvision.transforms as transforms

# =========================
# Audio
# =========================
import librosa

# =========================
# HuggingFace CLIP
# =========================
from transformers import CLIPModel, CLIPTokenizer

# =========================
# Evaluation
# =========================
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report
)

# =========================
# Utilities
# =========================
from tqdm import tqdm
import warnings
warnings.filterwarnings("ignore")

print("Libraries loaded successfully")

Libraries loaded successfully


In [2]:
# =========================
# Device
# =========================
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

# =========================
# Training settings
# =========================
BATCH_SIZE = 4        # safe for P100
NUM_EPOCHS = 10
LR = 1e-4

# =========================
# Video settings
# =========================
NUM_FRAMES = 16
IMG_SIZE = 224

# =========================
# CLIP settings
# =========================
MAX_TEXT_LEN = 77

# =========================
# Audio settings
# =========================
SR = 16000
N_MFCC = 40
MAX_AUDIO_LENGTH = 5  # seconds

# =========================
# Seed
# =========================
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print("Configuration initialized")

Device: cuda
Configuration initialized


In [3]:
# =========================
# Dataset Root
# =========================
DATA_ROOT = Path("/kaggle/input/datasets/ajfaisal002/crossmodal-misleading-video-dataset")

# Frames
TRAIN_FRAMES = DATA_ROOT / "dataset_kaggle/extracted_frames/train"
TEST_FRAMES = DATA_ROOT / "dataset_kaggle/extracted_frames/test"

# Audio
TRAIN_AUDIO = DATA_ROOT / "extracted_audio/extracted_audio/train"
TEST_AUDIO = DATA_ROOT / "extracted_audio/extracted_audio/test"

# Metadata CSV
TRAIN_META = DATA_ROOT / "dataset_kaggle/extracted_text/train_metadata.csv"
TEST_META = DATA_ROOT / "dataset_kaggle/extracted_text/test_metadata.csv"

print("Dataset paths loaded")

Dataset paths loaded


In [4]:
# =========================
# Load metadata
# =========================
train_df = pd.read_csv(TRAIN_META)
test_df = pd.read_csv(TEST_META)

print("Train samples:", len(train_df))
print("Test samples:", len(test_df))

train_df.head()

Train samples: 1600
Test samples: 800


,video_id,has_audio,transcript,ocr_text,category,subcategory
0,IF_001,True,আপনি সিদ্ধান্ত নিয়েছেন একটি নতুন অ্যাপ্লিকেশন...,OR EVEN TAKA ON YOUR FIRST CRAZY TIME,misleading,identity_fabrication
1,IF_002,True,"With a small investment of 30,500 Bangladeshi ...",YES WITH A SMALL INVESTMENT OF BANGLADESH TAKA...,misleading,identity_fabrication
2,IF_003,True,"If they close this page, they can't come back ...",IF THEY CLOSE THIS PAGE AS WILL THEIR CHANCE T...,misleading,identity_fabrication
3,IF_004,True,We tested our product with a small group of vo...,AND EACH OF THEM EARNED OVER BANGLADESHI TAKA ...,misleading,identity_fabrication
4,IF_005,True,I am ready to give you your money back from my...,MONEY BACK FROM MY POCKET BANGLADESHI TAKA,misleading,identity_fabrication


In [5]:
# =========================
# Label mapping
# =========================
LABEL_MAP = {
    "safe": 0,
    "identity_fabrication": 1,
    "perception_manipulation": 2,
    "scientifically_unrealistic_scene": 3,
    "surreal_content": 4
}

ID2LABEL = {v:k for k,v in LABEL_MAP.items()}

NUM_CLASSES = len(LABEL_MAP)

print(LABEL_MAP)

{'safe': 0, 'identity_fabrication': 1, 'perception_manipulation': 2, 'scientifically_unrealistic_scene': 3, 'surreal_content': 4}


In [6]:
# =========================
# Frame transform
# =========================
frame_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.48145466, 0.4578275, 0.40821073],
        std=[0.26862954, 0.26130258, 0.27577711]
    )
])

In [7]:
# =========================
# MFCC Feature Extraction
# =========================

def extract_mfcc(audio_path, sr=16000, n_mfcc=40, max_length=5):

    try:
        y, sr = librosa.load(audio_path, sr=sr)

        max_samples = sr * max_length

        if len(y) > max_samples:
            y = y[:max_samples]
        else:
            padding = max_samples - len(y)
            y = np.pad(y, (0, padding))

        mfcc = librosa.feature.mfcc(
            y=y,
            sr=sr,
            n_mfcc=n_mfcc
        )

        return mfcc

    except:
        return np.zeros((n_mfcc, 160))

In [8]:
# =========================
# Format MFCC tensor
# =========================

def process_audio(audio_path):

    mfcc = extract_mfcc(audio_path)

    target_length = 160

    if mfcc.shape[1] > target_length:
        mfcc = mfcc[:, :target_length]

    elif mfcc.shape[1] < target_length:
        pad_width = target_length - mfcc.shape[1]
        mfcc = np.pad(mfcc, ((0,0),(0,pad_width)))

    mfcc = torch.tensor(mfcc, dtype=torch.float32)

    return mfcc

In [9]:
# =========================
# CLIP tokenizer
# =========================

tokenizer = CLIPTokenizer.from_pretrained("openai/clip-vit-base-patch32")

print("Tokenizer loaded")

tokenizer_config.json:   0%|          | 0.00/592 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

Tokenizer loaded


In [10]:
# =========================
# Build label
# =========================

def get_label(row):

    if row["category"] == "safe":
        return LABEL_MAP["safe"]

    return LABEL_MAP[row["subcategory"]]

In [11]:
class MultimodalVideoDataset(Dataset):

    def __init__(self, df, frame_dir, audio_dir):

        self.df = df.reset_index(drop=True)
        self.frame_dir = frame_dir
        self.audio_dir = audio_dir


    def load_frames(self, video_id):

        video_path = self.frame_dir / video_id

        frames = []

        for i in range(1, NUM_FRAMES + 1):

            frame_path = video_path / f"frame_{i:02d}.jpg"

            if frame_path.exists():

                img = Image.open(frame_path).convert("RGB")
                img = frame_transform(img)

            else:

                img = torch.zeros(3,224,224)

            frames.append(img)

        frames = torch.stack(frames)

        return frames


    def load_audio(self, video_id, has_audio):

        if not has_audio:
            return torch.zeros(N_MFCC,160)

        audio_path = self.audio_dir / f"{video_id}.wav"

        if not audio_path.exists():
            return torch.zeros(N_MFCC,160)

        return process_audio(audio_path)


    def load_text(self, transcript, ocr):
        transcript = "" if pd.isna(transcript) else transcript
        ocr = "" if pd.isna(ocr) else ocr
    
        text = transcript + " " + ocr
    
        encoded = tokenizer(
            text,
            padding="max_length",
            truncation=True,
            max_length=MAX_TEXT_LEN,
            return_tensors="pt"
        )
    
        return {
            "input_ids": encoded["input_ids"].squeeze(0),
            "attention_mask": encoded["attention_mask"].squeeze(0),
        }


    def __len__(self):
        return len(self.df)


    def __getitem__(self, idx):

        row = self.df.iloc[idx]

        video_id = row["video_id"]

        frames = self.load_frames(video_id)

        audio = self.load_audio(video_id, row["has_audio"])

        text = self.load_text(
            row["transcript"],
            row["ocr_text"]
        )

        label = get_label(row)

        return {
          "frames": frames,
          "audio": audio,
          "text_input_ids": text["input_ids"],
          "text_attention_mask": text["attention_mask"],
          "label": torch.tensor(label)
        }

In [12]:
train_dataset = MultimodalVideoDataset(
    train_df,
    TRAIN_FRAMES,
    TRAIN_AUDIO
)

test_dataset = MultimodalVideoDataset(
    test_df,
    TEST_FRAMES,
    TEST_AUDIO
)

print("Datasets created")

Datasets created


In [13]:
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print("Dataloaders ready")

Dataloaders ready


In [14]:
sample = train_dataset[0]

print("Frames shape:", sample["frames"].shape)
print("Audio shape:", sample["audio"].shape)
# print("Text shape:", sample["text"].shape)
print("Label:", sample["label"])

Frames shape: torch.Size([16, 3, 224, 224])
Audio shape: torch.Size([40, 160])
Label: tensor(1)


In [15]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

clip = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device)

# Freeze CLIP encoders to save memory
for p in clip.parameters():
    p.requires_grad = False

clip.eval()
print("CLIP loaded + frozen.")

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/605M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
vision_model.embeddings.position_ids | UNEXPECTED |  | 
text_model.embeddings.position_ids   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


CLIP loaded + frozen.


In [16]:
def clip_vision_embed(pixel_values):
    """
    Always returns a TENSOR embedding, compatible across HF versions.
    pixel_values: (N,3,224,224)
    """
    outputs = clip.vision_model(pixel_values=pixel_values)

    if hasattr(outputs, "pooler_output") and outputs.pooler_output is not None:
        return outputs.pooler_output  # (N, D)

    return outputs.last_hidden_state.mean(dim=1)  # fallback (N, D)


def clip_text_embed(input_ids, attention_mask):
    """
    Always returns a TENSOR embedding, compatible across HF versions.
    input_ids: (B,77), attention_mask: (B,77)
    """
    outputs = clip.text_model(input_ids=input_ids, attention_mask=attention_mask)

    if hasattr(outputs, "pooler_output") and outputs.pooler_output is not None:
        return outputs.pooler_output  # (B, D)

    return outputs.last_hidden_state.mean(dim=1)  # fallback (B, D)

In [17]:
class AudioEncoder(nn.Module):
    def __init__(self, out_dim=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(16, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, 3, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((4, 4)),
        )
        self.fc = nn.Linear(64 * 4 * 4, out_dim)

    def forward(self, x):
        # x: (B, 40, 160)
        x = x.unsqueeze(1)            # (B,1,40,160)
        x = self.net(x)               # (B,64,4,4)
        x = x.view(x.size(0), -1)     # (B,1024)
        return self.fc(x)             # (B,128)

In [18]:
batch0 = next(iter(train_loader))

frames0 = batch0["frames"]                      # (B,16,3,224,224)
audio0  = batch0["audio"]                       # (B,40,160)
ids0    = batch0["text_input_ids"]              # (B,77)
mask0   = batch0["text_attention_mask"]         # (B,77)

B = frames0.size(0)

frames0_flat = frames0.view(-1, 3, 224, 224).to(device)  # (B*16,3,224,224)
audio0 = audio0.to(device)
ids0 = ids0.to(device)
mask0 = mask0.to(device)

audio_encoder0 = AudioEncoder(out_dim=128).to(device)
audio_encoder0.eval()

with torch.no_grad():
    img_feat0 = clip_vision_embed(frames0_flat)          # (B*16, Dimg)
    img_feat0 = img_feat0.view(B, 16, -1).mean(dim=1)    # (B, Dimg)

    txt_feat0 = clip_text_embed(ids0, mask0)             # (B, Dtxt)

    aud_feat0 = audio_encoder0(audio0)                   # (B, 128)

fusion_dim = img_feat0.shape[1] + txt_feat0.shape[1] + aud_feat0.shape[1]
print("Detected dims -> image:", img_feat0.shape[1], 
      "text:", txt_feat0.shape[1], 
      "audio:", aud_feat0.shape[1],
      "fusion:", fusion_dim)

model.safetensors:   0%|          | 0.00/605M [00:00<?, ?B/s]

Detected dims -> image: 768 text: 512 audio: 128 fusion: 1408


In [19]:
class FusionClassifierVTA(nn.Module):
    def __init__(self, fusion_dim, num_classes=5, audio_out=128):
        super().__init__()
        self.audio_encoder = AudioEncoder(out_dim=audio_out)

        self.classifier = nn.Sequential(
            nn.Linear(fusion_dim, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, num_classes)
        )

    def forward(self, frames, input_ids, attention_mask, audio):
        b = frames.size(0)

        # frames: (B,16,3,224,224)
        frames = frames.view(-1, 3, 224, 224).to(device)

        # audio: (B,40,160)
        audio = audio.to(device)

        input_ids = input_ids.to(device)
        attention_mask = attention_mask.to(device)

        # frozen CLIP embeddings
        with torch.no_grad():
            img_feat = clip_vision_embed(frames)        # (B*16, Dimg)
        img_feat = img_feat.view(b, 16, -1).mean(dim=1) # (B, Dimg)

        with torch.no_grad():
            txt_feat = clip_text_embed(input_ids, attention_mask)  # (B, Dtxt)

        # trainable audio embedding
        aud_feat = self.audio_encoder(audio)            # (B, 128)

        fusion = torch.cat([img_feat, txt_feat, aud_feat], dim=1)  # (B, fusion_dim)

        return self.classifier(fusion)

In [20]:
model = FusionClassifierVTA(fusion_dim=fusion_dim, num_classes=5).to(device)
print("Fusion V+T+A model created.")

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    list(model.audio_encoder.parameters()) + list(model.classifier.parameters()),
    lr=1e-3
)

scaler = GradScaler("cuda")
print("Optimizer created. Trainable params:",
      sum(p.numel() for p in model.parameters() if p.requires_grad))

Fusion V+T+A model created.
Optimizer created. Trainable params: 878469


In [21]:
batch = next(iter(train_loader))
frames = batch["frames"]
audio  = batch["audio"]
ids    = batch["text_input_ids"]
mask   = batch["text_attention_mask"]

model.eval()
with torch.no_grad():
    out = model(frames, ids, mask, audio)

print(out.shape)  # expected: (BATCH_SIZE, 5)

torch.Size([4, 5])


In [22]:
from torch.cuda.amp import autocast, GradScaler

scaler = GradScaler()   # Kaggle-safe

EPOCHS = 5

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0.0

    for batch in tqdm(train_loader):
        frames = batch["frames"]
        audio  = batch["audio"]
        ids    = batch["text_input_ids"]
        mask   = batch["text_attention_mask"]
        labels = batch["label"].to(device)

        optimizer.zero_grad(set_to_none=True)

        with autocast():   # ✅ no device_type here
            outputs = model(frames, ids, mask, audio)
            loss = criterion(outputs, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}/{EPOCHS} Loss: {total_loss/len(train_loader):.4f}")

100%|██████████| 400/400 [03:06<00:00,  2.14it/s]


Epoch 1/5 Loss: 0.4384


100%|██████████| 400/400 [02:20<00:00,  2.86it/s]


Epoch 2/5 Loss: 0.1855


100%|██████████| 400/400 [02:17<00:00,  2.92it/s]


Epoch 3/5 Loss: 0.1233


100%|██████████| 400/400 [02:17<00:00,  2.90it/s]


Epoch 4/5 Loss: 0.1246


100%|██████████| 400/400 [02:16<00:00,  2.92it/s]

Epoch 5/5 Loss: 0.0406


In [23]:
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
import numpy as np

@torch.no_grad()
def evaluate_vta(model, loader):
    model.eval()

    all_preds = []
    all_labels = []

    for batch in tqdm(loader):
        frames = batch["frames"]
        audio  = batch["audio"]
        ids    = batch["text_input_ids"]
        mask   = batch["text_attention_mask"]
        labels = batch["label"].to(device)

        logits = model(frames, ids, mask, audio)
        preds = torch.argmax(logits, dim=1)

        all_preds.append(preds.cpu().numpy())
        all_labels.append(labels.cpu().numpy())

    all_preds = np.concatenate(all_preds)
    all_labels = np.concatenate(all_labels)

    acc = accuracy_score(all_labels, all_preds)
    wf1 = f1_score(all_labels, all_preds, average="weighted")
    mf1 = f1_score(all_labels, all_preds, average="macro")

    return acc, wf1, mf1, all_labels, all_preds

In [24]:
acc, wf1, mf1, y_true, y_pred = evaluate_vta(model, test_loader)

print("Test Accuracy:", acc)
print("Test Weighted F1:", wf1)
print("Test Macro F1:", mf1)

label_names = ["safe", "identity_fabrication", "perception_manipulation",
               "scientifically_unrealistic_scene", "surreal_content"]

print("\nClassification Report:\n")
print(classification_report(y_true, y_pred, target_names=label_names, digits=4))

100%|██████████| 200/200 [01:28<00:00,  2.26it/s]


Test Accuracy: 0.7675
Test Weighted F1: 0.7591187900456176
Test Macro F1: 0.7167916863209602

Classification Report:

                                  precision    recall  f1-score   support

                            safe     0.7732    0.8950    0.8297       400
            identity_fabrication     0.7727    0.5100    0.6145       100
         perception_manipulation     0.7681    0.5300    0.6272       100
scientifically_unrealistic_scene     0.6800    0.8500    0.7556       100
                 surreal_content     0.8701    0.6700    0.7571       100

                        accuracy                         0.7675       800
                       macro avg     0.7728    0.6910    0.7168       800
                    weighted avg     0.7730    0.7675    0.7591       800



In [25]:
import pandas as pd

cm = confusion_matrix(y_true, y_pred)
cm_df = pd.DataFrame(cm, index=label_names, columns=label_names)
cm_df

,safe,identity_fabrication,perception_manipulation,scientifically_unrealistic_scene,surreal_content
safe,358,3,3,26,10
identity_fabrication,43,51,4,2,0
perception_manipulation,29,11,53,7,0
scientifically_unrealistic_scene,7,1,7,85,0
surreal_content,26,0,2,5,67


In [26]:
def vision_features(frames):
    # frames: (B,16,3,224,224) on CPU
    b = frames.size(0)
    frames = frames.view(-1, 3, 224, 224).to(device)

    with torch.no_grad():
        img_feat = clip_vision_embed(frames)              # (B*16, Dimg)
    img_feat = img_feat.view(b, 16, -1).mean(dim=1)       # (B, Dimg)
    return img_feat


def text_features(input_ids, attention_mask):
    input_ids = input_ids.to(device)
    attention_mask = attention_mask.to(device)

    with torch.no_grad():
        txt_feat = clip_text_embed(input_ids, attention_mask)  # (B, Dtxt)
    return txt_feat

In [27]:
@torch.no_grad()
def logits_vision_only(frames):
    # frames: (B,16,3,224,224)
    b = frames.size(0)
    frames = frames.view(-1, 3, 224, 224).to(device)
    img_feat = clip_vision_embed(frames)            # (B*16, D)
    img_feat = img_feat.view(b, 16, -1).mean(dim=1) # (B, D)

    # lightweight classifier head (we will define it below)
    return vision_head(img_feat)


@torch.no_grad()
def logits_text_only(input_ids, attention_mask):
    txt_feat = clip_text_embed(input_ids.to(device), attention_mask.to(device))
    return text_head(txt_feat)


@torch.no_grad()
def logits_vision_text(frames, input_ids, attention_mask):
    b = frames.size(0)
    frames = frames.view(-1, 3, 224, 224).to(device)

    img_feat = clip_vision_embed(frames)
    img_feat = img_feat.view(b, 16, -1).mean(dim=1)

    txt_feat = clip_text_embed(input_ids.to(device), attention_mask.to(device))

    fusion = torch.cat([img_feat, txt_feat], dim=1)
    return vt_head(fusion)

In [28]:
batch0 = next(iter(train_loader))
v0 = vision_features(batch0["frames"])
t0 = text_features(batch0["text_input_ids"], batch0["text_attention_mask"])

vision_dim = v0.shape[1]
text_dim = t0.shape[1]

print("vision_dim:", vision_dim, "text_dim:", text_dim)

vision_head = nn.Linear(vision_dim, 5).to(device)
text_head   = nn.Linear(text_dim, 5).to(device)
vt_head     = nn.Linear(vision_dim + text_dim, 5).to(device)

vision_dim: 768 text_dim: 512


In [29]:
from torch.cuda.amp import autocast, GradScaler

def train_head(head, mode="vision", epochs=2, lr=1e-3):
    head.train()
    opt = optim.Adam(head.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss()
    scaler = GradScaler()

    for ep in range(epochs):
        total = 0.0
        for batch in tqdm(train_loader):
            labels = batch["label"].to(device)

            opt.zero_grad(set_to_none=True)

            # Extract frozen features (no grad) + train head (grad)
            if mode == "vision":
                feats = vision_features(batch["frames"])              # (B, Dv)
            elif mode == "text":
                feats = text_features(batch["text_input_ids"], batch["text_attention_mask"])  # (B, Dt)
            elif mode == "vt":
                v = vision_features(batch["frames"])
                t = text_features(batch["text_input_ids"], batch["text_attention_mask"])
                feats = torch.cat([v, t], dim=1)                      # (B, Dv+Dt)
            else:
                raise ValueError("Unknown mode")

            with autocast():
                logits = head(feats)   # <-- gradients flow here
                loss = loss_fn(logits, labels)

            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()

            total += loss.item()

        print(f"[{mode}] Epoch {ep+1}/{epochs} Loss: {total/len(train_loader):.4f}")

In [30]:
@torch.no_grad()
def eval_head(head=None, mode="vision"):
    all_preds, all_labels = [], []

    for batch in tqdm(test_loader):
        labels = batch["label"].to(device)

        if mode == "vision":
            feats = vision_features(batch["frames"])
            logits = head(feats)
        elif mode == "text":
            feats = text_features(batch["text_input_ids"], batch["text_attention_mask"])
            logits = head(feats)
        elif mode == "vt":
            v = vision_features(batch["frames"])
            t = text_features(batch["text_input_ids"], batch["text_attention_mask"])
            feats = torch.cat([v, t], dim=1)
            logits = head(feats)
        elif mode == "vta":
            logits = model(batch["frames"], batch["text_input_ids"], batch["text_attention_mask"], batch["audio"])
        else:
            raise ValueError("Unknown mode")

        preds = torch.argmax(logits, dim=1)

        all_preds.append(preds.cpu().numpy())
        all_labels.append(labels.cpu().numpy())

    all_preds = np.concatenate(all_preds)
    all_labels = np.concatenate(all_labels)

    acc = accuracy_score(all_labels, all_preds)
    wf1 = f1_score(all_labels, all_preds, average="weighted")
    mf1 = f1_score(all_labels, all_preds, average="macro")

    return acc, wf1, mf1

In [31]:
# Train heads
train_head(vision_head, mode="vision", epochs=2)
train_head(text_head, mode="text", epochs=2)
train_head(vt_head, mode="vt", epochs=2)

# Evaluate
results = {}
results["Vision only"] = eval_head(vision_head, "vision")
results["Text only"] = eval_head(text_head, "text")
results["Vision + Text"] = eval_head(vt_head, "vt")
results["Vision + Text + Audio"] = eval_head(None, "vta")

res_df = pd.DataFrame(results, index=["Accuracy", "Weighted F1", "Macro F1"]).T
res_df

100%|██████████| 400/400 [02:14<00:00,  2.97it/s]


[vision] Epoch 1/2 Loss: 0.5225


100%|██████████| 400/400 [02:12<00:00,  3.02it/s]


[vision] Epoch 2/2 Loss: 0.2592


100%|██████████| 400/400 [02:07<00:00,  3.13it/s]


...

[vt] Epoch 1/2 Loss: 0.4197


100%|██████████| 400/400 [02:13<00:00,  2.99it/s]


[vt] Epoch 2/2 Loss: 0.1352


100%|██████████| 200/200 [01:07<00:00,  2.95it/s]


,Accuracy,Weighted F1,Macro F1
Vision only,0.74000,0.731534,0.681308
Text only,0.68250,0.666885,0.579692
Vision + Text,0.74375,0.730609,0.665526
Vision + Text + Audio,0.76750,0.759119,0.716792
